# 2. Radiomic features

Runs on Colab without a GPU.

**Needs** the outputs of notebook 1 and `Liver-Train.zip` in `MyDrive`.

**Makes** (in `MyDrive/CARE2026_ckpts`)
- `lifs_stad_features/stad_features.csv`: first version of the shape, texture, appearance and direction (STAD) features
- `lifs_stad_enriched.csv`: adds nine shape and intensity descriptors
- `lifs_stad_ged4_clean.csv`: GED4 and shape features only
- `lifs_stad_fixed.csv`: the 198 features used in the paper, from all seven series after fixing the extraction error (Sec. 4.2 of the paper), plus the nine descriptors

`stad_features.csv` and `lifs_stad_enriched.csv` come from the crops with the extraction error. CELL C in notebook 4 uses `lifs_stad_enriched.csv` for Table 2, which shows the error.

The STAD features follow Zhou et al. (CARE 2025): https://github.com/YangForever/care2025_liver_biodreamer

**Step 1.** Mount Google Drive, install SimpleITK, set the project folder and extract `Liver-Train.zip` (same code as the start of Step 6). In our sessions notebook 1 and an install cell had already done this in the same Colab runtime.

In [ ]:
# Settings: mount Google Drive, install SimpleITK, set the project folder and extract the training data
from google.colab import drive
drive.mount('/content/drive')
!pip install -q SimpleITK scikit-image 2>&1 | tail -1
import os, glob, time, shutil, zipfile
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"
DATA = "/content/Liver-Train-data/Liver-Train"
if len(glob.glob(f"{DATA}/Vendor_*/*/GED4.nii.gz")) < 460:
    t0 = time.time()
    shutil.copy("/content/drive/MyDrive/Liver-Train.zip", "/content/Liver-Train.zip")
    print(f"zip copied ({time.time()-t0:.0f}s)")
    with zipfile.ZipFile("/content/Liver-Train.zip") as z:
        z.extractall("/content/Liver-Train-data", members=[n for n in z.namelist() if "__MACOSX" not in n])
    os.remove("/content/Liver-Train.zip")
    print(f"extracted ({time.time()-t0:.0f}s)")
print("GED4 files:", len(glob.glob(f"{DATA}/Vendor_*/*/GED4.nii.gz")))

**Step 2.** STAD features from the crops in `lifs_roi_cache` (first version, with the extraction error).

In [ ]:
import os, time, json, warnings, shutil
import numpy as np
import pandas as pd
from scipy import ndimage
from skimage.feature import graycomatrix, graycoprops
warnings.filterwarnings("ignore")

ROI_DIR = "/content/drive/MyDrive/CARE2026_ckpts/lifs_roi_cache"
FEAT_DIR = "/content/drive/MyDrive/CARE2026_ckpts/lifs_stad_features"
os.makedirs(FEAT_DIR, exist_ok=True)
PHASES = ['T1', 'T2', 'DWI_800', 'GED1', 'GED2', 'GED3', 'GED4']

def shape_features(mask):
    vox = mask.sum()
    if vox == 0:
        return [0.0] * 7
    zz, yy, xx = np.where(mask > 0)
    dz = zz.max() - zz.min() + 1
    dy = yy.max() - yy.min() + 1
    dx = xx.max() - xx.min() + 1
    bbox_vol = dz * dy * dx
    extent = vox / max(bbox_vol, 1)
    struct = ndimage.generate_binary_structure(3, 1)
    eroded = ndimage.binary_erosion(mask > 0, struct)
    surface = (mask > 0).sum() - eroded.sum()
    compactness = surface ** 1.5 / max(vox, 1) if vox > 0 else 0.0
    dims = sorted([dz, dy, dx])
    elongation = dims[1] / max(dims[2], 1)
    flatness = dims[0] / max(dims[2], 1)
    return [float(vox), float(surface), float(extent), float(compactness),
            float(elongation), float(flatness), float(bbox_vol)]

def firstorder_features(vals):
    if vals.size == 0:
        return [0.0] * 10
    mu = vals.mean(); sd = vals.std()
    mn = vals.min(); mx = vals.max()
    p10, p25, p50, p75, p90 = np.percentile(vals, [10, 25, 50, 75, 90])
    if sd > 1e-8:
        skew = ((vals - mu) ** 3).mean() / (sd ** 3)
        kurt = ((vals - mu) ** 4).mean() / (sd ** 4)
    else:
        skew = 0.0; kurt = 0.0
    hist, _ = np.histogram(vals, bins=32)
    p = hist / max(hist.sum(), 1)
    p = p[p > 0]
    entropy = -(p * np.log2(p)).sum() if p.size > 0 else 0.0
    energy = float((vals ** 2).sum())
    iqr = p75 - p25
    return [float(mu), float(sd), float(mn), float(mx), float(p50),
            float(skew), float(kurt), float(entropy), float(iqr), float(energy)]

def texture_features(vol, mask):
    m = mask > 0
    if m.sum() < 10:
        return [0.0] * 10
    vals = vol[m]
    lo, hi = np.percentile(vals, [1, 99])
    if hi - lo < 1e-6:
        return [0.0] * 10
    nbins = 16
    q = np.clip((vol - lo) / (hi - lo), 0, 1)
    q = (q * (nbins - 1)).astype(np.uint8)
    feats = {'contrast': [], 'homogeneity': [], 'energy': [], 'correlation': [], 'dissimilarity': []}
    zc = np.where(m.any(axis=(1, 2)))[0]
    if len(zc) == 0:
        return [0.0] * 10
    step = max(1, len(zc) // 8)
    for zi in zc[::step]:
        sl = q[zi].copy()
        slm = m[zi]
        if slm.sum() < 10:
            continue
        sl[~slm] = 0
        try:
            glcm = graycomatrix(sl, distances=[1], angles=[0, np.pi/2],
                                levels=nbins, symmetric=True, normed=True)
            for k in feats:
                feats[k].append(graycoprops(glcm, k).mean())
        except Exception:
            continue
    out = []
    for k in ['contrast', 'homogeneity', 'energy', 'correlation', 'dissimilarity']:
        arr = feats[k]
        out.append(float(np.mean(arr)) if arr else 0.0)
        out.append(float(np.std(arr)) if arr else 0.0)
    return out

def directional_features(vol, mask):
    m = mask > 0
    if m.sum() < 10:
        return [0.0] * 6
    gz, gy, gx = np.gradient(vol)
    gmag = np.sqrt(gz ** 2 + gy ** 2 + gx ** 2)
    gm = gmag[m]
    sob = ndimage.sobel(vol, axis=0) ** 2 + ndimage.sobel(vol, axis=1) ** 2 + ndimage.sobel(vol, axis=2) ** 2
    sob = np.sqrt(sob)[m]
    ax_energy = [float((g[m] ** 2).mean()) for g in (gz, gy, gx)]
    anisotropy = max(ax_energy) / (min(ax_energy) + 1e-8)
    return [float(gm.mean()), float(gm.std()), float(sob.mean()),
            float(anisotropy), float(ax_energy[0]), float(ax_energy[2])]

def extract_case(npz_path):
    d = np.load(npz_path, allow_pickle=True)
    vol = d['vol']; mask = d['mask']; avail = d['avail']
    feats = []
    feats.extend(shape_features(mask))
    for ci, ph in enumerate(PHASES):
        if avail[ci] > 0:
            v = vol[ci]; m = mask > 0
            vals = v[m]
            feats.extend(firstorder_features(vals))
            feats.extend(texture_features(v, mask))
            feats.extend(directional_features(v, mask))
        else:
            feats.extend([np.nan] * (10 + 10 + 6))
    return np.array(feats, dtype=np.float32), str(d['vendor']), int(d['stage']), str(d['filename'])

def feature_names():
    names = ['shape_vox', 'shape_surf', 'shape_extent', 'shape_compact',
             'shape_elong', 'shape_flat', 'shape_bbox']
    fo = ['mean', 'std', 'min', 'max', 'median', 'skew', 'kurt', 'entropy', 'iqr', 'energy']
    tx = ['contrast_m', 'contrast_s', 'homog_m', 'homog_s', 'energy_m', 'energy_s',
          'corr_m', 'corr_s', 'dissim_m', 'dissim_s']
    dr = ['grad_mean', 'grad_std', 'sobel_mean', 'aniso', 'ax_z', 'ax_x']
    for ph in PHASES:
        for n in fo + tx + dr:
            names.append(f"{ph}_{n}")
    return names

df = pd.read_csv("/content/Liver-Train-data/Liver-Train/info_2026.csv")
cases = df['filename'].astype(str).tolist()
out_path = f"{FEAT_DIR}/stad_features.csv"

rows = []
t0 = time.time()
for i, fn in enumerate(cases):
    p = f"{ROI_DIR}/{fn}.npz"
    if not os.path.exists(p):
        print(f"  missing roi: {fn}", flush=True)
        continue
    try:
        feat, vendor, stage, name = extract_case(p)
        rows.append([name, vendor, stage] + feat.tolist())
    except Exception as e:
        print(f"  ERROR {fn}: {e}", flush=True)
    if (i + 1) % 50 == 0:
        el = time.time() - t0
        print(f"[{i+1}/{len(cases)}] {el/(i+1):.2f}s/case", flush=True)

cols = ['filename', 'vendor', 'stage'] + feature_names()
feat_df = pd.DataFrame(rows, columns=cols)
local = "/content/_stad.csv"
feat_df.to_csv(local, index=False)
shutil.copy(local, out_path)
print(f"saved {feat_df.shape} to {out_path}")
print("feature columns:", len(cols) - 3)

**Step 3.** Extract the GED4 files.

In [ ]:
# === re-extract GED4 from Liver-Train.zip ===
import zipfile, os, glob, time
zp="/content/drive/MyDrive/Liver-Train.zip"; dst="/content/Liver-Train"
os.makedirs(dst,exist_ok=True)
t0=time.time()
with zipfile.ZipFile(zp) as z:
    members=[n for n in z.namelist() if n.endswith("GED4.nii.gz")]
    print(f"extracting {len(members)} GED4 files...")
    for i,m in enumerate(members,1):
        z.extract(m,dst)
        if i%100==0: print(f"  {i}/{len(members)} ({time.time()-t0:.0f}s)")
g=glob.glob(f"{dst}/**/GED4.nii.gz",recursive=True)
print(f"✓ {len(g)} GED4 files in {time.time()-t0:.0f}s")

**Step 4.** Add the nine descriptors (`lifs_stad_enriched.csv`).

In [ ]:
# === Step 2 FULL: enrich STAD with vendor-robust shape features ===
import numpy as np, glob, os, pandas as pd, SimpleITK as sitk, time
from skimage import measure
from scipy.spatial import ConvexHull
from scipy.stats import skew, kurtosis

CKPT="/content/drive/MyDrive/CARE2026_ckpts"
RAW="/content/Liver-Train"
MASK_DIRS=[f"{CKPT}/lifs_train_masks_extra", f"{CKPT}/lifs_train_masks_st2_345"]
OLD_STAD=pd.read_csv(f"{CKPT}/lifs_stad_features/stad_features.csv")

# build case -> mask path map
mask_map={}
for md in MASK_DIRS:
    for f in glob.glob(f"{md}/*.nii.gz"):
        mask_map[os.path.basename(f).replace(".nii.gz","")]=f
print(f"masks mapped: {len(mask_map)}")

def raw_ged4(case):
    h=glob.glob(f"{RAW}/**/{case}/GED4.nii.gz",recursive=True)
    return h[0] if h else None

def extra_shape_features(img_arr, mask_arr):
    feats={}
    m = mask_arr > 0
    if m.sum() < 10: return None
    if img_arr.shape != m.shape: return "SHAPE_MISMATCH"
    try:
        verts,faces,_,_=measure.marching_cubes(m.astype(float), level=0.5)
        tris=verts[faces]                       # (n_faces,3,3)
        v1=tris[:,1]-tris[:,0]; v2=tris[:,2]-tris[:,0]
        area=float(0.5*np.linalg.norm(np.cross(v1,v2),axis=1).sum())
        feats['mc_surface']=area
        try:
            hull=ConvexHull(verts)
            feats['convex_vol']=float(hull.volume)
            feats['surf_to_hull']=float(area/(hull.area+1e-6))
        except:
            feats['convex_vol']=np.nan; feats['surf_to_hull']=np.nan
        feats['compactness2']=float((area**1.5)/(m.sum()+1e-6))
    except:
        feats['mc_surface']=np.nan; feats['convex_vol']=np.nan
        feats['surf_to_hull']=np.nan; feats['compactness2']=np.nan
    roi=img_arr[m]
    feats['roi_skew2']=float(skew(roi)); feats['roi_kurt2']=float(kurtosis(roi))
    feats['roi_p10']=float(np.percentile(roi,10)); feats['roi_p90']=float(np.percentile(roi,90))
    feats['roi_iqr2']=float(np.percentile(roi,75)-np.percentile(roi,25))
    return feats

print(f"computing enriched features for {len(mask_map)} cases...")
t0=time.time(); rows=[]; n_ok=0; n_shape_fail=0; n_mc_nan=0; n_no_raw=0
for i,(case,mp) in enumerate(sorted(mask_map.items()),1):
    gp=raw_ged4(case)
    if gp is None:
        n_no_raw+=1; continue
    img_itk=sitk.ReadImage(gp, sitk.sitkFloat32)
    img_arr=sitk.GetArrayFromImage(img_itk)
    mask_itk=sitk.ReadImage(mp)
    mask_r=sitk.Resample(mask_itk, img_itk, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    mask_arr=sitk.GetArrayFromImage(mask_r)

    ef=extra_shape_features(img_arr, mask_arr)
    if ef is None:
        continue
    if ef=="SHAPE_MISMATCH":
        n_shape_fail+=1
        if n_shape_fail<=3: print(f"  ⚠ shape mismatch: {case} img{img_arr.shape} mask{mask_arr.shape}")
        continue
    if np.isnan(ef['mc_surface']): n_mc_nan+=1
    ef['filename']=case
    rows.append(ef); n_ok+=1
    if i%50==0: print(f"  {i}/{len(mask_map)} (ok={n_ok}, {time.time()-t0:.0f}s)")

print(f"\n✓ ok={n_ok} | shape_fails={n_shape_fail} | mc_nan={n_mc_nan} | no_raw={n_no_raw} | {time.time()-t0:.0f}s")
if n_ok < 400:
    print("⚠️ WARNING: <400 cases succeeded — investigate before proceeding")

extra_df=pd.DataFrame(rows)
enriched=OLD_STAD.merge(extra_df, on='filename', how='left')
enriched.to_csv(f"{CKPT}/lifs_stad_enriched.csv", index=False)
print(f"\nenriched STAD: {enriched.shape} (was {OLD_STAD.shape})")
print("new features:", [c for c in extra_df.columns if c!='filename'])
print("\nNaN counts in new features:")
print(extra_df.drop(columns=['filename']).isna().sum())

**Step 5.** GED4 and shape features only (`lifs_stad_ged4_clean.csv`).

In [ ]:
# === CELL 2: rebuild clean GED4 radiomic features for all 460 ===
!pip install -q SimpleITK scikit-image 2>&1 | tail -1
import zipfile, os, glob, time, warnings
import numpy as np, pandas as pd, SimpleITK as sitk
from scipy import ndimage
from scipy.stats import spearmanr
from skimage.feature import graycomatrix, graycoprops
warnings.filterwarnings("ignore")

CKPT = "/content/drive/MyDrive/CARE2026_ckpts"
OUT  = f"{CKPT}/lifs_stad_ged4_clean.csv"
RAW  = "/content/Liver-Train"

def find_ged4():
    return {os.path.basename(os.path.dirname(p)): p
            for p in glob.glob(f"{RAW}/**/GED4.nii.gz", recursive=True)}
ged4 = find_ged4()
if len(ged4) < 460:
    t0 = time.time()
    with zipfile.ZipFile("/content/drive/MyDrive/Liver-Train.zip") as z:
        mem = [n for n in z.namelist() if n.endswith("/GED4.nii.gz") and "__MACOSX" not in n]
        for i, m in enumerate(mem, 1):
            z.extract(m, RAW)
            if i % 100 == 0: print(f"  extracted {i}/{len(mem)} ({time.time()-t0:.0f}s)")
    ged4 = find_ged4()
print(f"GED4 volumes: {len(ged4)}")

masks = {}
for d in [f"{CKPT}/lifs_train_masks_extra", f"{CKPT}/lifs_train_masks_st2_345"]:
    for p in glob.glob(f"{d}/*.nii.gz"):
        masks[os.path.basename(p)[:-7]] = p
print(f"ST2 masks: {len(masks)}")
lab = pd.read_csv(f"{CKPT}/lifs_stad_enriched.csv")

# ---- feature functions copied verbatim from the original STAD cell ----
def shape_features(mask):
    vox = mask.sum()
    if vox == 0: return [0.0] * 7
    zz, yy, xx = np.where(mask > 0)
    dz = zz.max()-zz.min()+1; dy = yy.max()-yy.min()+1; dx = xx.max()-xx.min()+1
    bbox_vol = dz*dy*dx; extent = vox / max(bbox_vol, 1)
    eroded = ndimage.binary_erosion(mask > 0, ndimage.generate_binary_structure(3, 1))
    surface = (mask > 0).sum() - eroded.sum()
    compactness = surface**1.5 / max(vox, 1) if vox > 0 else 0.0
    dims = sorted([dz, dy, dx])
    return [float(vox), float(surface), float(extent), float(compactness),
            float(dims[1]/max(dims[2],1)), float(dims[0]/max(dims[2],1)), float(bbox_vol)]

def firstorder_features(vals):
    if vals.size == 0: return [0.0] * 10
    mu = vals.mean(); sd = vals.std(); mn = vals.min(); mx = vals.max()
    p10, p25, p50, p75, p90 = np.percentile(vals, [10, 25, 50, 75, 90])
    if sd > 1e-8:
        skew = ((vals-mu)**3).mean()/(sd**3); kurt = ((vals-mu)**4).mean()/(sd**4)
    else:
        skew = 0.0; kurt = 0.0
    hist, _ = np.histogram(vals, bins=32); p = hist/max(hist.sum(), 1); p = p[p > 0]
    entropy = -(p*np.log2(p)).sum() if p.size > 0 else 0.0
    return [float(mu), float(sd), float(mn), float(mx), float(p50), float(skew),
            float(kurt), float(entropy), float(p75-p25), float((vals**2).sum())]

def texture_features(vol, mask):
    m = mask > 0
    if m.sum() < 10: return [0.0] * 10
    vals = vol[m]; lo, hi = np.percentile(vals, [1, 99])
    if hi - lo < 1e-6: return [0.0] * 10
    nbins = 16
    q = (np.clip((vol-lo)/(hi-lo), 0, 1)*(nbins-1)).astype(np.uint8)
    feats = {'contrast': [], 'homogeneity': [], 'energy': [], 'correlation': [], 'dissimilarity': []}
    zc = np.where(m.any(axis=(1, 2)))[0]
    if len(zc) == 0: return [0.0] * 10
    for zi in zc[::max(1, len(zc)//8)]:
        sl = q[zi].copy(); slm = m[zi]
        if slm.sum() < 10: continue
        sl[~slm] = 0
        try:
            glcm = graycomatrix(sl, distances=[1], angles=[0, np.pi/2], levels=nbins,
                                symmetric=True, normed=True)
            for k in feats: feats[k].append(graycoprops(glcm, k).mean())
        except Exception:
            continue
    out = []
    for k in ['contrast', 'homogeneity', 'energy', 'correlation', 'dissimilarity']:
        a = feats[k]; out += [float(np.mean(a)) if a else 0.0, float(np.std(a)) if a else 0.0]
    return out

def directional_features(vol, mask):
    m = mask > 0
    if m.sum() < 10: return [0.0] * 6
    gz, gy, gx = np.gradient(vol); gm = np.sqrt(gz**2+gy**2+gx**2)[m]
    sob = np.sqrt(ndimage.sobel(vol, 0)**2 + ndimage.sobel(vol, 1)**2 + ndimage.sobel(vol, 2)**2)[m]
    ax = [float((g[m]**2).mean()) for g in (gz, gy, gx)]
    return [float(gm.mean()), float(gm.std()), float(sob.mean()),
            float(max(ax)/(min(ax)+1e-8)), float(ax[0]), float(ax[2])]

SHAPE = ['shape_vox','shape_surf','shape_extent','shape_compact','shape_elong','shape_flat','shape_bbox']
FO = ['mean','std','min','max','median','skew','kurt','entropy','iqr','energy']
TX = ['contrast_m','contrast_s','homog_m','homog_s','energy_m','energy_s','corr_m','corr_s','dissim_m','dissim_s']
DR = ['grad_mean','grad_std','sobel_mean','aniso','ax_z','ax_x']
NAMES = SHAPE + [f"GED4_{n}" for n in FO + TX + DR]

# ---- ROI in array-index space (no physical-space resampling) ----
OUT_SHAPE = (64, 128, 128); MARGIN = 0.10
def fit(a, shape):
    o = np.zeros(shape, a.dtype); s = tuple(slice(0, min(a.shape[i], shape[i])) for i in range(3))
    o[s] = a[s]; return o

def roi(case):
    img = sitk.ReadImage(ged4[case], sitk.sitkFloat32)
    mk  = sitk.Resample(sitk.ReadImage(masks[case]), img, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    a, m = sitk.GetArrayFromImage(img), sitk.GetArrayFromImage(mk) > 0
    if m.sum() == 0: return None, None
    lo, hi = [], []
    for v, n in zip(np.where(m), a.shape):
        pad = int(round((v.max()-v.min()+1)*MARGIN))
        lo.append(max(0, v.min()-pad)); hi.append(min(n, v.max()+pad+1))
    sl = tuple(slice(lo[i], hi[i]) for i in range(3))
    a, m = a[sl], m[sl]
    z = [OUT_SHAPE[i]/a.shape[i] for i in range(3)]
    a = fit(ndimage.zoom(a, z, order=1), OUT_SHAPE)
    m = fit(ndimage.zoom(m.astype(np.uint8), z, order=0), OUT_SHAPE)
    mu, sd = a[m > 0].mean(), a[m > 0].std() + 1e-6
    return ((a-mu)/sd).astype(np.float32), m

rows, fails = [], []
t0 = time.time()
for i, r in enumerate(lab.itertuples(index=False), 1):
    c = r.filename
    try:
        v, m = roi(c)
        if v is None: fails.append((c, "empty mask")); continue
        feats = shape_features(m) + firstorder_features(v[m > 0]) \
              + texture_features(v, m) + directional_features(v, m)
        rows.append([c, r.vendor, r.stage] + feats)
    except Exception as e:
        fails.append((c, str(e)[:60]))
    if i % 50 == 0: print(f"  {i}/{len(lab)} ({time.time()-t0:.0f}s)")

clean = pd.DataFrame(rows, columns=['filename','vendor','stage'] + NAMES)
ENR = ['mc_surface','convex_vol','surf_to_hull','compactness2',
       'roi_skew2','roi_kurt2','roi_p10','roi_p90','roi_iqr2']
clean = clean.merge(lab[['filename'] + ENR], on='filename', how='left')
clean.to_csv(OUT, index=False)
print(f"\nsaved {clean.shape} -> {OUT} | failures: {len(fails)} {fails[:5]}")

F = clean[NAMES].values
print(f"all-zero rows now: {np.all(F == 0, axis=1).sum()}  (must be 0)")

is_b2 = clean['vendor'].astype(str).eq('B2').values
cols = NAMES + ENR
ID, B2 = clean[~is_b2], clean[is_b2]
r = {c: spearmanr(ID[c], ID['stage'], nan_policy='omit')[0] for c in cols}
top = sorted([c for c in cols if np.isfinite(r[c])], key=lambda c: -abs(r[c]))[:15]
flips = 0
print("\n--- sign flip on CLEAN features ---")
for c in top:
    rb = spearmanr(B2[c], B2['stage'], nan_policy='omit')[0]
    f = np.sign(r[c]) != np.sign(rb); flips += int(f)
    print(f"  {c:20s} ID {r[c]:+.2f}  B2 {rb:+.2f}  {'FLIP' if f else ''}")
print(f"  flips: {flips}/15")

**Step 6.** Rebuild the features of all seven series with the crop grid fixed, and add the nine descriptors (`lifs_stad_fixed.csv`).

In [ ]:
# === STEP 1: rebuild 7-phase STAD features with the ROI orientation bug fixed (CPU, ~40 min) ===
from google.colab import drive
drive.mount('/content/drive')
!pip install -q SimpleITK scikit-image 2>&1 | tail -1
import os, glob, time, shutil, zipfile, warnings
import numpy as np, pandas as pd, SimpleITK as sitk
from scipy import ndimage
from skimage.feature import graycomatrix, graycoprops
warnings.filterwarnings("ignore")

CKPT     = "/content/drive/MyDrive/CARE2026_ckpts"
MASK_DIR = f"{CKPT}/liseg_v1/lifs_masks"     # same ST2 masks the original cache used
OUT      = f"{CKPT}/lifs_stad_fixed.csv"
DATA     = "/content/Liver-Train-data/Liver-Train"
PHASES   = ['T1','T2','DWI_800','GED1','GED2','GED3','GED4']
OUT_SHAPE, MARGIN = (64, 128, 128), 0.10

# ---- 1. raw data, all 7 phases ----
if len(glob.glob(f"{DATA}/Vendor_*/*/GED4.nii.gz")) < 460:
    t0 = time.time()
    shutil.copy("/content/drive/MyDrive/Liver-Train.zip", "/content/Liver-Train.zip")
    print(f"zip copied ({time.time()-t0:.0f}s)")
    with zipfile.ZipFile("/content/Liver-Train.zip") as z:
        z.extractall("/content/Liver-Train-data", members=[n for n in z.namelist() if "__MACOSX" not in n])
    os.remove("/content/Liver-Train.zip")
    print(f"extracted ({time.time()-t0:.0f}s)")
print("GED4 files:", len(glob.glob(f"{DATA}/Vendor_*/*/GED4.nii.gz")))

# ---- 2. FIXED ROI: grid anchored at the mask's lowest index, along the mask's own axes ----
def grid_from_mask(mask_img):
    a = sitk.GetArrayFromImage(mask_img) > 0
    zz, yy, xx = np.where(a)
    if len(zz) == 0: return None
    lo = np.array([xx.min(), yy.min(), zz.min()]); hi = np.array([xx.max(), yy.max(), zz.max()])
    pad = np.array([int(round((hi[d]-lo[d]+1)*MARGIN)) for d in range(3)])
    size = np.array(mask_img.GetSize())
    lo = np.maximum(0, lo-pad); hi = np.minimum(size-1, hi+pad)
    origin = mask_img.TransformIndexToPhysicalPoint([int(v) for v in lo])
    extent = (hi-lo) * np.array(mask_img.GetSpacing())
    return origin, extent, mask_img.GetDirection()

def to_grid(img, origin, extent, direction, is_mask):
    out_size = [OUT_SHAPE[2], OUT_SHAPE[1], OUT_SHAPE[0]]
    sp = [extent[d]/out_size[d] for d in range(3)]; sp = [s if s > 1e-6 else 1.0 for s in sp]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputOrigin(origin); rs.SetOutputDirection(direction)
    rs.SetOutputSpacing(sp); rs.SetSize([int(v) for v in out_size])
    rs.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(img)

def normalize(vol, mask):
    m = mask > 0
    if m.sum() > 0: mu, sd = vol[m].mean(), vol[m].std() + 1e-6
    else:
        nz = vol > 0
        if nz.sum() == 0: return vol.astype(np.float32)
        mu, sd = vol[nz].mean(), vol[nz].std() + 1e-6
    return ((vol-mu)/sd).astype(np.float32)

# ---- 3. feature functions, verbatim from the original STAD cell ----
def shape_features(mask):
    vox = mask.sum()
    if vox == 0: return [0.0]*7
    zz, yy, xx = np.where(mask > 0)
    dz = zz.max()-zz.min()+1; dy = yy.max()-yy.min()+1; dx = xx.max()-xx.min()+1
    bbox_vol = dz*dy*dx; extent = vox/max(bbox_vol, 1)
    eroded = ndimage.binary_erosion(mask > 0, ndimage.generate_binary_structure(3, 1))
    surface = (mask > 0).sum() - eroded.sum()
    compactness = surface**1.5/max(vox, 1) if vox > 0 else 0.0
    dims = sorted([dz, dy, dx])
    return [float(vox), float(surface), float(extent), float(compactness),
            float(dims[1]/max(dims[2], 1)), float(dims[0]/max(dims[2], 1)), float(bbox_vol)]

def firstorder_features(vals):
    if vals.size == 0: return [0.0]*10
    mu = vals.mean(); sd = vals.std(); mn = vals.min(); mx = vals.max()
    p10, p25, p50, p75, p90 = np.percentile(vals, [10, 25, 50, 75, 90])
    if sd > 1e-8: skew = ((vals-mu)**3).mean()/(sd**3); kurt = ((vals-mu)**4).mean()/(sd**4)
    else: skew = 0.0; kurt = 0.0
    hist, _ = np.histogram(vals, bins=32); p = hist/max(hist.sum(), 1); p = p[p > 0]
    entropy = -(p*np.log2(p)).sum() if p.size > 0 else 0.0
    return [float(mu), float(sd), float(mn), float(mx), float(p50), float(skew),
            float(kurt), float(entropy), float(p75-p25), float((vals**2).sum())]

def texture_features(vol, mask):
    m = mask > 0
    if m.sum() < 10: return [0.0]*10
    vals = vol[m]; lo, hi = np.percentile(vals, [1, 99])
    if hi-lo < 1e-6: return [0.0]*10
    nb = 16; q = (np.clip((vol-lo)/(hi-lo), 0, 1)*(nb-1)).astype(np.uint8)
    feats = {'contrast': [], 'homogeneity': [], 'energy': [], 'correlation': [], 'dissimilarity': []}
    zc = np.where(m.any(axis=(1, 2)))[0]
    if len(zc) == 0: return [0.0]*10
    for zi in zc[::max(1, len(zc)//8)]:
        sl = q[zi].copy(); slm = m[zi]
        if slm.sum() < 10: continue
        sl[~slm] = 0
        try:
            g = graycomatrix(sl, distances=[1], angles=[0, np.pi/2], levels=nb, symmetric=True, normed=True)
            for k in feats: feats[k].append(graycoprops(g, k).mean())
        except Exception: continue
    out = []
    for k in ['contrast', 'homogeneity', 'energy', 'correlation', 'dissimilarity']:
        a = feats[k]; out += [float(np.mean(a)) if a else 0.0, float(np.std(a)) if a else 0.0]
    return out

def directional_features(vol, mask):
    m = mask > 0
    if m.sum() < 10: return [0.0]*6
    gz, gy, gx = np.gradient(vol); gm = np.sqrt(gz**2+gy**2+gx**2)[m]
    sob = np.sqrt(ndimage.sobel(vol, 0)**2+ndimage.sobel(vol, 1)**2+ndimage.sobel(vol, 2)**2)[m]
    ax = [float((g[m]**2).mean()) for g in (gz, gy, gx)]
    return [float(gm.mean()), float(gm.std()), float(sob.mean()),
            float(max(ax)/(min(ax)+1e-8)), float(ax[0]), float(ax[2])]

FO = ['mean','std','min','max','median','skew','kurt','entropy','iqr','energy']
TX = ['contrast_m','contrast_s','homog_m','homog_s','energy_m','energy_s','corr_m','corr_s','dissim_m','dissim_s']
DR = ['grad_mean','grad_std','sobel_mean','aniso','ax_z','ax_x']
NAMES = ['shape_vox','shape_surf','shape_extent','shape_compact','shape_elong','shape_flat','shape_bbox'] \
      + [f"{ph}_{n}" for ph in PHASES for n in FO+TX+DR]

# ---- 4. extract ----
info = pd.read_csv(f"{DATA}/info_2026.csv")
rows, empty_grid, fails = [], [], []
t0 = time.time()
for i, r in enumerate(info.itertuples(index=False), 1):
    fn, vendor = r.filename, r.vendor
    try:
        mk = sitk.ReadImage(f"{MASK_DIR}/{fn}.nii.gz", sitk.sitkUInt8)
        g = grid_from_mask(mk)
        if g is None: fails.append((fn, "empty source mask")); continue
        m = (sitk.GetArrayFromImage(to_grid(mk, *g, True)) > 0).astype(np.uint8)
        if m.sum() == 0: empty_grid.append(fn)
        feats = shape_features(m)
        for ph in PHASES:
            p = f"{DATA}/Vendor_{vendor}/{fn}/{ph}.nii.gz"
            if not os.path.exists(p): feats += [np.nan]*26; continue
            try:
                v = normalize(sitk.GetArrayFromImage(to_grid(sitk.ReadImage(p, sitk.sitkFloat32), *g, False)), m)
                feats += firstorder_features(v[m > 0]) + texture_features(v, m) + directional_features(v, m)
            except Exception:
                feats += [np.nan]*26
        rows.append([fn, vendor, int(str(fn).split('-S')[-1])] + feats)
    except Exception as e:
        fails.append((fn, str(e)[:60]))
    if i % 50 == 0: print(f"  {i}/{len(info)} ({time.time()-t0:.0f}s)")

fixed = pd.DataFrame(rows, columns=['filename', 'vendor', 'stage'] + NAMES)

# ---- 5. checks ----
print(f"\ncases: {len(fixed)} | empty crop grids: {len(empty_grid)} (must be 0) | failures: {len(fails)} {fails[:3]}")
F = fixed[NAMES].values.astype(float)
print(f"all-zero rows: {np.all((F == 0) | np.isnan(F), axis=1).sum()} (must be 0)")

old = pd.read_csv(f"{CKPT}/lifs_stad_features/stad_features.csv").set_index('filename')
Fo = old[NAMES].values.astype(float)
good = old.index[~np.all((Fo == 0) | np.isnan(Fo), axis=1)]
a = fixed.set_index('filename').reindex(good)[NAMES].values.astype(float)
b = old.loc[good, NAMES].values.astype(float)
rel = np.abs(a-b)/(np.abs(b)+1e-6)
print(f"previously-working {len(good)} cases, old vs fixed: median rel diff {np.nanmedian(rel):.4f}, "
      f"95th pct {np.nanpercentile(rel, 95):.4f}  (expect ~0)")

# ---- 6. add the 9 enriched features (already clean) and save ----
ENR = ['mc_surface','convex_vol','surf_to_hull','compactness2','roi_skew2','roi_kurt2','roi_p10','roi_p90','roi_iqr2']
enr = pd.read_csv(f"{CKPT}/lifs_stad_enriched.csv")[['filename'] + ENR]
fixed = fixed.merge(enr, on='filename', how='left')
fixed.to_csv(OUT, index=False)
print(f"saved {fixed.shape} -> {OUT}")

**Step 7.** How much the fix changes the features of the examinations whose crops were not empty: median rank correlation 0.98 (Sec. 4.2). The change in standard deviation units printed here is a different measure from the relative change in the paper, which Step 6 prints.

In [ ]:
import numpy as np, pandas as pd
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"
old = pd.read_csv(f"{CKPT}/lifs_stad_features/stad_features.csv").set_index("filename")
new = pd.read_csv(f"{CKPT}/lifs_stad_fixed.csv").set_index("filename")
cols = [c for c in old.columns if c not in ("vendor", "stage")]
Fo = old[cols].values.astype(float)
good = old.index[~np.all((Fo == 0) | np.isnan(Fo), axis=1)]
a, b = new.loc[good, cols], old.loc[good, cols]
z = ((a - b).abs() / b.std()).values                      # change in units of between-patient SD
r = [a[c].corr(b[c], method="spearman") for c in cols]
print(f"change in SD units: median {np.nanmedian(z):.4f}, 95th {np.nanpercentile(z, 95):.4f}, 99th {np.nanpercentile(z, 99):.4f}")
print(f"per-feature rank correlation old vs fixed: median {np.nanmedian(r):.4f}, min {np.nanmin(r):.4f}")
print("lowest 5:", sorted(zip(np.round(r, 3), cols))[:5])